In [ ]:
import os

import pandas as pd
import torch


image size: 28x28
pixelvalue: 0-255


In [ ]:
data_path = r"..\..\data\kaggle\digit-recognizer"
train_path = os.path.join(data_path, "train.csv")
test_path = os.path.join(data_path, "test.csv")

df_train = pd.read_csv(train_path)
df_test = pd.read_csv(test_path)
df_train_label = df_train["label"]
df_train = df_train.drop(columns=["label"])
df_train.shape, df_train_label.shape, df_test.shape


In [ ]:
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

train_images = torch.tensor(df_train.to_numpy(), dtype=torch.float32).to(device)
train_images.div_(255.0)
train_images = train_images.view(-1, 1, 28, 28)
train_labels = torch.tensor(df_train_label.to_numpy(), dtype=torch.int64).to(device)
test_images = torch.tensor(df_test.to_numpy(), dtype=torch.float32).to(device)
test_images.div_(255.0)
test_images = test_images.view(-1, 1, 28, 28)

print(train_images.shape, train_labels.shape, test_images.shape)

In [ ]:
from torch import nn


class CNNModel(nn.Module):
    def __init__(self, H, W, C, L) -> None:
        super().__init__()
        self.H = H  # height
        self.W = W  # width
        self.C = C  # channel
        self.L = L  # labels

        self.net = nn.Sequential(
            nn.Conv2d(self.C, 32, kernel_size=(3, 3), bias=True, padding=1),
            nn.ReLU(),
            nn.MaxPool2d((2, 2), stride=2),
            nn.Conv2d(32, 64, kernel_size=(3, 3), padding=1),
            nn.ReLU(),
            nn.MaxPool2d((2, 2), stride=2),
            nn.Flatten(start_dim=1),
            nn.Linear(in_features=(self.H // 4) * (self.W // 4) * 64, out_features=128),
            nn.ReLU(),
            nn.Linear(128, 10),
        )

    def forward(self, images: torch.Tensor):
        return self.net(images)

In [ ]:
H = 28  # image height
W = 28  # image width
C = 1  # image channels
L = 10  # num of labels
num_batches = 20

model = CNNModel(H, W, C, L).to(device)

dataset = torch.utils.data.TensorDataset(train_images, train_labels)
dataloader = torch.utils.data.DataLoader(
    dataset, batch_size=train_images.size(0) // num_batches, shuffle=True
)

In [ ]:
num_epochs = 1000
lr = 1e-4
lossfunc = nn.CrossEntropyLoss(reduction="mean")
optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=0)
model.train()
for epoch in range(num_epochs):
    accumloss = 0
    for batch_id, (batch_images, batch_labels) in enumerate(dataloader):
        pred_labels = model(batch_images)
        loss = lossfunc(pred_labels, batch_labels)
        loss.backward()
        optimizer.step()
        accumloss += loss.item()
    print(epoch, accumloss)

In [ ]:
import torch.nn.functional as F

model.eval()
with torch.inference_mode():
    test_logits = model(test_images)
    test_logits = F.softmax(test_logits, dim=1)
    test_logits = test_logits.argmax(dim=1)
    test_labels = test_logits.cpu().detach().numpy()
    num_test_images = len(test_labels)
    df_results = pd.DataFrame({"ImageId": range(1, num_test_images + 1), "Label": test_labels})
    df_results.to_csv("submission.csv", index=False)